# L01_WCE_SQRT_T02B_R50 v1.0 — CE có trọng số 1/√n_c (RUN_TRAIN=False)

So sánh: **T02B_D4_R50** (ResNet50 + D4, CE không trọng số — đối chứng trực tiếp) vs **T02B + CE có trọng số**. Không dùng SoG làm đối chứng.

Thay đổi DUY NHẤT: loss train `CrossEntropyLoss(weight=w, reduction='mean', label_smoothing=0)`, `w_c ∝ 1/sqrt(n_c)` với `n_c` từ **train manifest**, trung bình 8 trọng số = 1. Validation loss vẫn là CE **không trọng số**. Train loss mới có định nghĩa khác — không so độ lớn với T02B.

Giả thuyết: có thể thay đổi validation macro-F1. Chưa chạy, chưa được gọi là cải thiện.

Mọi kiểm tra thất bại -> `STOP_REPORT_TO_CHA`. Không mở ảnh test. **Run All**; cell TRAIN bỏ qua khi RUN_TRAIN=False. Input cần gắn: Output T02AB (`reference.json`, `T02B_D4_R50/…`) và dataset ảnh ISIC 2019. GPU Tesla T4 + Internet (tải ImageNet V1) khi train.


In [1]:
from pathlib import Path
import os, json, random, hashlib, shutil, time, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, balanced_accuracy_score
from IPython.display import display

cv2.setNumThreads(1)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

import inspect, gc


Device: cuda


In [2]:
import inspect, gc, platform, math, difflib
# ===== L01_WCE_SQRT_T02B_R50 v1.0 — PREFLIGHT + TRAIN (chỉ thay loss train) =====
RUN_TRAIN = True  # Mặc định: chỉ preflight. Chỉ đổi True sau khi CHA duyệt preflight_audit.json.
RUN_TEST = False    # Không bao giờ bật trong L01. Notebook không tạo đường dẫn ảnh test.
assert RUN_TEST is False
METHOD = 'L01_WCE_SQRT_T02B_R50'
CONTROL = 'T02B_D4_R50'
INPUT_ROOT = Path('/kaggle/input')
OUT_ROOT = Path('/kaggle/working/ISIC2019_R50_L01_WCE_SQRT_v1_0')   # thư mục mới, không ghi đè output cũ
OUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT = OUT_ROOT
resume = None

LABELS = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 16
PATIENCE = 4
NUM_WORKERS = 2
LR = 1e-4
WEIGHT_DECAY = 1e-4
HAIR = dict(kernel=9, threshold=20, min_area=8, max_area=500,
            min_length=12, min_elongation=4.0, max_width=5.0,
            max_fraction=0.03, radius=2)   # Chỉ để khớp hash preprocess; nhánh B00/T02B không dùng.
EXPECTED = {
 "split": "38a5a9aca907d56685bddfcd874e162d8bf2c141a7247025410eeeee6bcf522a",
 "b00_lock": "c598d6fb6835ea831c8c169233768f1c2ed860206356b5c09a7164dd8099edec",
 "t02b_lock": "47eacd88da997d3df482a2682f515847e5acb442b1257fd34e65f80eefab3559",
 "t02b_source": "ce274d6b2dcb00dc3c1a2604a14e6920e4af5f549865ab1f04e56699fa93f9bc",
 "training_source": "8399596159991d8fbafecf72d511fa4f1775cdc8e43f1ff299614c0ec66ac22c",
 "initial_sha": "37a03a16d86fda7f927000366760327e605b16fc8b4cb6a565a4ce763fe9003c",
 "t02_files": {
  "split_manifest.csv": "38a5a9aca907d56685bddfcd874e162d8bf2c141a7247025410eeeee6bcf522a",
  "B00_protocol_lock.json": "c598d6fb6835ea831c8c169233768f1c2ed860206356b5c09a7164dd8099edec",
  "B00_complete.json": "8201f09b2d3b305cc1c2beb227652c8a249e6e6eb24ec12e0ab86be670949a2b",
  "reference.json": "9d3cd0fdf9e4891c7bd5076476e6f7c0c9ed201efed53ba8ff3d32061e862779",
  "B00_val_predictions.csv": "d002ba18195a95c8aa546d6c6c50a711a4343e91b9ecf56abf3223390a7be225",
  "analysis_source.py": "e0f07a2218d5ec2c6f321a4b379cbb2ba622ea34ef76d6f5dd47bbbe667d0a4a",
  "T02B_D4_R50/candidate_protocol_lock.json": "47eacd88da997d3df482a2682f515847e5acb442b1257fd34e65f80eefab3559",
  "T02B_D4_R50/candidate_source.py": "ce274d6b2dcb00dc3c1a2604a14e6920e4af5f549865ab1f04e56699fa93f9bc",
  "T02B_D4_R50/T02B_D4_R50/complete.json": "1b3d0a01d8376d11206db78a7310f3e16dec41cf12d0925aca2b6b94ab6ac672",
  "T02B_D4_R50/T02B_D4_R50/val_predictions.csv": "41a406228c3654282b0710346e66c437837c8e1d92117db6ee6933eab6f6774b"
 },
 "ce_files": {
  "class_distribution.csv": "d95825b4c25fc9d9962aed9944b22704c8ce10c625413ada878b2b5936360b27",
  "data_audit.json": "b7dd9b5830e61d7f1aee7c6d02d4ade5f0a26070ceafb2443eb9db0516c8bb37"
 }
}
label_to_id = {l: i for i, l in enumerate(LABELS)}
AUDIT_FILE = 'preflight_audit.json'   # sau preflight đổi sang run_audit.json
AUDIT = dict(task_id='L01_WCE_SQRT_T02B_R50', version='1.0', phase='preflight', method=METHOD, control=CONTROL,
             evidence_rule='Mọi check trong file này chạy trên Kaggle với dữ liệu thật (kaggle_real_data).',
             status='RUNNING', checks={})

def stop(name, detail=None):
    AUDIT['status'] = 'STOP_REPORT_TO_CHA'
    AUDIT['stop_reason'] = dict(check=name, detail=str(detail)[:2000])
    (OUT_ROOT / AUDIT_FILE).write_text(json.dumps(AUDIT, ensure_ascii=False, indent=2, default=str))
    raise RuntimeError(f'STOP_REPORT_TO_CHA: {name}: {detail}')

def check(name, ok, detail=None):
    AUDIT['checks'][name] = dict(status='PASS' if ok else 'FAIL', detail=detail, evidence='kaggle_real_data')
    print(('PASS ' if ok else 'FAIL ') + name, '' if detail is None else str(detail)[:300])
    if not ok:
        stop(name, detail)

def sha_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def dump_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')

def reset_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

reset_seed()

# Hash mã train có guard NaN/Inf (đã dùng ở P04SOG v1.1; hash gốc 'training_source' giữ nguyên).
EXPECTED['training_source_guarded'] = '2a10a660d2a4d1304914e344c110c1a2cf3440fbfb49c2c29df946f41fe53f82'

# ===== L01: giá trị kỳ vọng tính từ split_manifest.csv 38a5a9ac… (CHỈ split=train) =====
EXPECTED['t02b_extra'] = {
 'T02B_D4_R50/T02B_D4_R50/history.csv': 'a58f2b866d5bf81095d6d9d2904889840706b3869b4cbac32bc80811c0bf7bf5',
 'T02B_D4_R50/T02B_D4_R50/val_per_class.csv': 'd72e6f1891378ce16ef25d96f0aa110d0fbd0f71c27a1e93897aa7e4984041e2',
 'T02B_D4_R50/T02B_D4_R50/val_confusion.csv': '98301d70d06aa4140bc6725c0f6013bb89b9a783d580e6856ef26a95d30fffbc',
 'T02B_D4_R50/T02B_D4_R50/val_metrics.json': 'bba570b85662784a26fa8f06797b152ac9d5c768fdb21f8c4a3dc46f901b368a',
 'bootstrap_group_support.csv': '9683f41ccf7280280249302b98f10e38babc504efc56cde6c9dd8001918035bb',
}
EXPECTED['n_train'] = 17723
EXPECTED['n_val'] = 2539
EXPECTED['val_groups'] = 1392
EXPECTED['train_counts'] = {"MEL": 3077, "NV": 9019, "BCC": 2374, "AK": 611, "BKL": 1847, "DF": 187, "VASC": 151, "SCC": 457}
EXPECTED['class_weights'] = {'MEL': 0.4591693890646419, 'NV': 0.26819920266119224, 'BCC': 0.5227527408409827, 'AK': 1.0304242497848128, 'BKL': 0.5926567404688351, 'DF': 1.8625840198638355, 'VASC': 2.072756101347629, 'SCC': 1.1914575559680705}


In [3]:
# Tìm Output T02 đã gắn (notebook output hoặc dataset tải lên từ 'results (9).zip').
roots = sorted(p.parent for p in INPUT_ROOT.rglob('reference.json')
               if (p.parent / 'T02B_D4_R50' / 'candidate_protocol_lock.json').is_file())
check('t02_output_found', len(roots) >= 1, [str(r) for r in roots])
T02_IN = roots[0]
for rel, sha in EXPECTED['t02_files'].items():
    hashes = sorted({sha_file(r / rel) if (r / rel).is_file() else 'MISSING' for r in roots})
    check(f'sha:{rel}', hashes == [sha], hashes)
ce_roots = sorted(p.parent for p in INPUT_ROOT.rglob('class_distribution.csv') if (p.parent / 'data_audit.json').is_file())
CE_IN = ce_roots[0] if ce_roots else None
if CE_IN is not None:
    for rel, sha in EXPECTED['ce_files'].items():
        check(f'sha:CE/{rel}', sha_file(CE_IN / rel) == sha, sha_file(CE_IN / rel))
lock = json.loads((T02_IN / 'B00_protocol_lock.json').read_text())
reference = json.loads((T02_IN / 'reference.json').read_text())
cand_lock = json.loads((T02_IN / 'T02B_D4_R50/candidate_protocol_lock.json').read_text())
b00_result = json.loads((T02_IN / 'B00_complete.json').read_text())
t02b_result = json.loads((T02_IN / 'T02B_D4_R50/T02B_D4_R50/complete.json').read_text())
check('reference_points_to_lock_split', reference['baseline_lock_sha'] == EXPECTED['b00_lock'] and reference['split_sha'] == EXPECTED['split'])
check('candidate_lock_embeds_B00_lock', cand_lock['baseline_config'] == lock and cand_lock['reference'] == reference)
check('T02B_complete_lock_init', t02b_result['lock_sha'] == EXPECTED['t02b_lock'] and t02b_result['initial_sha'] == EXPECTED['initial_sha'] == b00_result['initial_sha'])
# Checkpoint B00/T02B: KHÔNG cần. Không tải, không train tiếp từ chúng.
AUDIT['checkpoint_dependency'] = ('Không cần best_resnet50.pt của B00/T02B. Notebook gốc chỉ dùng sha checkpoint B00 để '
    'xác nhận danh tính output B00; tương đương ở đây: sha B00_complete.json (chứa checkpoint_sha), lock, split, '
    'B00_val_predictions và initial_sha tái lập từ ImageNet V1.')


PASS t02_output_found ['/kaggle/input/datasets/tuanducnguyen123/output-fork-of-isic-2019-resnet50-t02ab/ISIC2019_R50_T02_AB_v1', '/kaggle/input/datasets/tuanducnguyen123/output-fork-of-isic-2019-resnet50-t02ab/ISIC2019_R50_T02_AB_v1_reports']
PASS sha:split_manifest.csv ['38a5a9aca907d56685bddfcd874e162d8bf2c141a7247025410eeeee6bcf522a']
PASS sha:B00_protocol_lock.json ['c598d6fb6835ea831c8c169233768f1c2ed860206356b5c09a7164dd8099edec']
PASS sha:B00_complete.json ['8201f09b2d3b305cc1c2beb227652c8a249e6e6eb24ec12e0ab86be670949a2b']
PASS sha:reference.json ['9d3cd0fdf9e4891c7bd5076476e6f7c0c9ed201efed53ba8ff3d32061e862779']
PASS sha:B00_val_predictions.csv ['d002ba18195a95c8aa546d6c6c50a711a4343e91b9ecf56abf3223390a7be225']
PASS sha:analysis_source.py ['e0f07a2218d5ec2c6f321a4b379cbb2ba622ea34ef76d6f5dd47bbbe667d0a4a']
PASS sha:T02B_D4_R50/candidate_protocol_lock.json ['47eacd88da997d3df482a2682f515847e5acb442b1257fd34e65f80eefab3559']
PASS sha:T02B_D4_R50/candidate_source.py ['ce274d6b2

In [4]:
env_now = dict(torch=torch.__version__, torchvision=torchvision.__version__, numpy=np.__version__,
               pandas=pd.__version__, sklearn=__import__('sklearn').__version__, cv2=cv2.__version__,
               pillow=Image.__version__ if hasattr(Image, '__version__') else __import__('PIL').__version__,
               gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
env_lock = lock['environment']
diff = {k: (env_lock[k], env_now[k]) for k in env_lock if k != 'gpu' and env_lock[k] != env_now[k]}
check('environment_libraries_equal_lock', not diff, diff or env_now)
AUDIT['environment'] = env_now
if RUN_TRAIN:
    check('gpu_equals_lock', env_now['gpu'] == env_lock['gpu'], env_now['gpu'])
else:
    AUDIT['gpu_note'] = 'Preflight chạy được trên CPU; GPU Tesla T4 bắt buộc khi RUN_TRAIN=True.'
cfg = dict(labels=LABELS, seed=SEED, size=IMG_SIZE, batch=BATCH_SIZE, epochs=EPOCHS, patience=PATIENCE,
           workers=NUM_WORKERS, lr=LR, weight_decay=WEIGHT_DECAY, hair=HAIR)
check('notebook_config_equals_lock', all(lock[k] == v for k, v in cfg.items()), {k: (lock[k], v) for k, v in cfg.items() if lock[k] != v})


PASS environment_libraries_equal_lock {'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128', 'numpy': '2.1.3', 'pandas': '2.3.3', 'sklearn': '1.6.1', 'cv2': '4.14.0', 'pillow': '12.3.0', 'gpu': 'Tesla T4'}
PASS gpu_equals_lock Tesla T4
PASS notebook_config_equals_lock {}


In [5]:
manifest = pd.read_csv(T02_IN / 'split_manifest.csv', keep_default_na=False)
check('manifest_rows_columns', len(manifest) == 25327 and list(manifest.columns) == ['image','label','label_name','lesion_id','group','file_sha256','split'], (len(manifest), list(manifest.columns)))
check('label_order', (manifest.label_name == manifest.label.map(dict(enumerate(LABELS)))).all())
for c in ['image', 'file_sha256', 'group']:
    n = int((manifest.groupby(c).split.nunique() > 1).sum())
    check(f'no_cross_split_{c}', n == 0, n)
has_lesion = manifest.lesion_id.astype(str).str.strip().ne('')
n = int((manifest[has_lesion].groupby('lesion_id').split.nunique() > 1).sum())
check('no_cross_split_lesion_id', n == 0, dict(with_lesion_id=int(has_lesion.sum()), missing_lesion_id=int((~has_lesion).sum()), crossing=n))
AUDIT['leakage_limits'] = ('ISIC 2019 không có patient_id; 2084 ảnh thiếu lesion_id là group đơn (MISSING_<image>) '
    'nên không phát hiện được cùng tổn thương/cùng bệnh nhân; chỉ trùng file chính xác (SHA256) được gộp group; '
    'ảnh gần trùng (resize/crop/nén khác) chưa được loại trừ.')
parts = {s: manifest[manifest.split == s].sort_values('image').reset_index(drop=True) for s in ['train', 'val', 'test']}
# Test: chỉ giữ định danh để kiểm leakage. Không gắn path, không mở ảnh.
parts['test'] = parts['test'][['image', 'group', 'file_sha256', 'split']]
t02b_pred = pd.read_csv(T02_IN / 'T02B_D4_R50/T02B_D4_R50/val_predictions.csv', keep_default_na=False).sort_values('image').reset_index(drop=True)
v = parts['val'].rename(columns={'label_name': 'true_label'})
ok = list(t02b_pred.image) == list(v.image) and all((t02b_pred[c].values == v[c].values).all() for c in ['group', 'file_sha256', 'true_label'])
check('val_manifest_matches_T02B_predictions', ok, len(t02b_pred))

# Chỉ lập chỉ mục ảnh train+val. Tên file test bị bỏ qua khi quét, không bao giờ mở.
need = set(parts['train'].image) | set(parts['val'].image)
cand_paths = {}
for p in INPUT_ROOT.rglob('*.jpg'):
    if p.stem in need:
        cand_paths.setdefault(p.stem, []).append(p)
want = pd.concat([parts['train'], parts['val']])
def resolve(row):
    for p in sorted(cand_paths.get(row.image, [])):
        if sha_file(p) == row.file_sha256:
            return str(p)
    return None
with ThreadPoolExecutor(8) as ex:
    resolved = list(ex.map(resolve, [r for r in want.itertuples()]))
path_of = dict(zip(want.image, resolved))
bad = [k for k, p in path_of.items() if p is None]
check('train_val_image_sha256_match_manifest', not bad, dict(n=len(path_of), missing_or_mismatch=bad[:20], n_bad=len(bad)))
for s in ['train', 'val']:
    parts[s]['path'] = parts[s].image.map(path_of)
ALLOWED_PATHS = frozenset(path_of.values())
check('test_has_no_path_and_not_indexed', 'path' not in parts['test'].columns and not (set(parts['test'].image) & set(path_of)))


PASS manifest_rows_columns (25327, ['image', 'label', 'label_name', 'lesion_id', 'group', 'file_sha256', 'split'])
PASS label_order 
PASS no_cross_split_image 0
PASS no_cross_split_file_sha256 0
PASS no_cross_split_group 0
PASS no_cross_split_lesion_id {'with_lesion_id': 23243, 'missing_lesion_id': 2084, 'crossing': 0}
PASS val_manifest_matches_T02B_predictions 2539
PASS train_val_image_sha256_match_manifest {'n': 20262, 'missing_or_mismatch': [], 'n_bad': 0}
PASS test_has_no_path_and_not_indexed 


## Mã B00 chép nguyên văn + khớp hash `training_source` và `preprocessing_source`

In [6]:
# Chép NGUYÊN VĂN từ notebook T02AB (sha 9bb1ec46...00be). Không sửa một ký tự.
def preprocess(image, method):
    image = image.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
    rgb = np.asarray(image).copy()
    mask = np.zeros(rgb.shape[:2], dtype=np.uint8)
    if method == 'B00_R50':
        return image, mask, False
    assert method == 'A1_R50'
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (HAIR['kernel'], HAIR['kernel']))
    response = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    raw = (response > HAIR['threshold']).astype(np.uint8)
    count, components, stats, _ = cv2.connectedComponentsWithStats(raw, 8)
    for k in range(1, count):
        area = stats[k, cv2.CC_STAT_AREA]
        if not HAIR['min_area'] <= area <= HAIR['max_area']:
            continue
        yy, xx = np.where(components == k)
        points = np.column_stack([xx, yy]).astype(np.float32)
        (_, _), (w, h), _ = cv2.minAreaRect(points)
        length, width = max(w, h) + 1, min(w, h) + 1
        if length >= HAIR['min_length'] and width <= HAIR['max_width'] and length / width >= HAIR['min_elongation']:
            mask[components == k] = 255
    if np.any(mask):
        mask = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=1)
    fallback = float((mask > 0).mean()) > HAIR['max_fraction']
    if fallback or not np.any(mask):
        return image, mask, fallback
    repaired = cv2.inpaint(rgb, mask, HAIR['radius'], cv2.INPAINT_TELEA)
    return Image.fromarray(repaired), mask, fallback

normalize = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class SkinDataset(Dataset):
    def __init__(self, data, method):
        self.data = data.reset_index(drop=True)
        self.method = method
    def __len__(self):
        return len(self.data)
    def __getitem__(self, index):
        row = self.data.iloc[index]
        with Image.open(row.path) as im:
            image, _, _ = preprocess(im, self.method)
        return normalize(image), int(row.label)

def seed_worker(worker_id):
    seed = torch.initial_seed() % (2**32)
    random.seed(seed); np.random.seed(seed)

def make_loader(data, method, train=False):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(SkinDataset(data, method), batch_size=BATCH_SIZE,
        shuffle=train, num_workers=NUM_WORKERS, pin_memory=True,
        worker_init_fn=seed_worker, generator=generator, drop_last=False)

def build_model():
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, len(LABELS))
    return model.to(DEVICE)

criterion = nn.CrossEntropyLoss()  # Không weight; label_smoothing mặc định 0.

def metrics(y, pred):
    return dict(accuracy=float(accuracy_score(y, pred)),
        macro_f1=float(f1_score(y, pred, labels=range(8), average='macro', zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(y, pred)))

def evaluate(model, loader):
    model.eval()
    loss_sum, total, labels, probabilities = 0.0, 0, [], []
    with torch.no_grad():
        for images, y in loader:
            images, y = images.to(DEVICE), y.to(DEVICE)
            logits = model(images)
            loss_sum += criterion(logits, y).item() * len(y)
            total += len(y)
            labels.extend(y.cpu().tolist())
            probabilities.append(logits.softmax(1).cpu().numpy())
    probabilities = np.concatenate(probabilities)
    pred = probabilities.argmax(1)
    return loss_sum / total, metrics(labels, pred), np.array(labels), pred, probabilities

def export_evaluation(model, data, method, folder, split):
    loss, scores, y, pred, probs = evaluate(model, make_loader(data, method))
    result = data[['image', 'group', 'file_sha256', 'label_name']].rename(columns={'label_name':'true_label'}).copy()
    result['pred_label'] = [LABELS[i] for i in pred]
    for i, label in enumerate(LABELS): result['p_' + label] = probs[:, i]
    result.to_csv(folder / f'{split}_predictions.csv', index=False)
    report = classification_report(y, pred, labels=range(8), target_names=LABELS, output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(folder / f'{split}_per_class.csv')
    cm = confusion_matrix(y, pred, labels=range(8))
    pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(folder / f'{split}_confusion.csv')
    fig, ax = plt.subplots(figsize=(7, 6)); ax.imshow(cm, cmap='Blues')
    ax.set(xticks=range(8), yticks=range(8), xticklabels=LABELS, yticklabels=LABELS,
           xlabel='Dự đoán', ylabel='Nhãn thật', title=f'{method} — {split}')
    for i in range(8):
        for j in range(8): ax.text(j, i, str(cm[i,j]), ha='center', va='center', color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=8)
    fig.tight_layout(); fig.savefig(folder / f'{split}_confusion.png', dpi=140); plt.close(fig)
    dump_json(folder / f'{split}_metrics.json', dict(loss=loss, **scores))
    return scores

def train_experiment(method):
    assert DEVICE.type == 'cuda', 'Bật GPU; không train ResNet50 trên CPU.'
    folder = OUT / method
    if not folder.exists() and resume and (resume / method / 'complete.json').exists():
        shutil.copytree(resume / method, folder)
    folder.mkdir(exist_ok=True)
    complete_path = folder / 'complete.json'
    if complete_path.exists():
        saved = json.loads(complete_path.read_text())
        assert saved['lock_sha'] == LOCK_SHA
        assert sha_file(folder / 'best_resnet50.pt') == saved['checkpoint_sha']
        print(method, 'đã hoàn thành; tái sử dụng kết quả, không train lại.')
        return saved
    if (folder / 'history.csv').exists():
        raise RuntimeError('Lượt train trước chưa hoàn thành. Giữ output đó; dùng OUT mới để chạy lại cả protocol, không âm thầm resume optimizer.')
    reset_seed()
    model = build_model()
    # Hash khởi tạo: xác minh hai lượt khởi tạo giống nhau.
    initial = hashlib.sha256()
    for key, value in model.state_dict().items():
        initial.update(key.encode()); initial.update(value.detach().cpu().numpy().tobytes())
    init_sha = initial.hexdigest()
    train_loader = make_loader(parts['train'], method, train=True)
    val_loader = make_loader(parts['val'], method)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best, wait, history = -1.0, 0, []
    started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train(); loss_sum = 0.0; total = 0
        for images, y in train_loader:
            images, y = images.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, y)
            loss.backward(); optimizer.step()
            loss_sum += loss.item() * len(y); total += len(y)
        val_loss, scores, _, _, _ = evaluate(model, val_loader)
        history.append(dict(epoch=epoch, train_loss=loss_sum/total, val_loss=val_loss,
                            lr=optimizer.param_groups[0]['lr'], **scores))
        pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
        print(f'{method} epoch {epoch:02d}: train={loss_sum/total:.4f}, val={val_loss:.4f}, F1={scores["macro_f1"]:.4f}')
        if scores['macro_f1'] > best + 1e-5:
            best, best_epoch, wait = scores['macro_f1'], epoch, 0
            torch.save(model.state_dict(), folder / 'best_resnet50.pt')
        else:
            wait += 1
        scheduler.step()
        if wait >= PATIENCE: break
    model.load_state_dict(torch.load(folder / 'best_resnet50.pt', map_location=DEVICE, weights_only=True))
    scores = export_evaluation(model, parts['val'], method, folder, 'val')
    h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(h.epoch, h.train_loss, label='train'); axes[0].plot(h.epoch, h.val_loss, label='val'); axes[0].legend()
    axes[1].plot(h.epoch, h.macro_f1); axes[1].set_title('Validation macro-F1')
    for ax in axes: ax.set_xlabel('Epoch'); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(folder / 'learning_curves.png', dpi=120); plt.close(fig)
    saved = dict(method=method, lock_sha=LOCK_SHA, initial_sha=init_sha,
        checkpoint_sha=sha_file(folder / 'best_resnet50.pt'), best_epoch=best_epoch,
        completed_epochs=len(history), seconds=time.perf_counter()-started, **scores)
    dump_json(complete_path, saved)
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()
    return saved

src = '\n'.join(inspect.getsource(f) for f in [reset_seed, SkinDataset.__init__, SkinDataset.__getitem__, SkinDataset.__len__,
      seed_worker, make_loader, build_model, metrics, evaluate, export_evaluation, train_experiment])
check('training_source_sha', hashlib.sha256(src.encode()).hexdigest() == EXPECTED['training_source'], hashlib.sha256(src.encode()).hexdigest())
check('preprocess_source_sha', hashlib.sha256(inspect.getsource(preprocess).encode()).hexdigest() == lock['preprocessing_source'])


PASS training_source_sha 8399596159991d8fbafecf72d511fa4f1775cdc8e43f1ff299614c0ec66ac22c
PASS preprocess_source_sha 


In [7]:
# Chép NGUYÊN VĂN hàm T02 (t02_preflight/run_t02 chỉ để khớp hash, KHÔNG gọi).
METHODS_T02 = ('T02A_HV_R50', 'T02B_D4_R50')

def t02_geometry(rgb, method, code):
    assert rgb.shape == (IMG_SIZE, IMG_SIZE, 3)
    if method == 'T02A_HV_R50':
        assert code in range(4)
        if code & 1: rgb = np.flip(rgb, axis=1)
        if code & 2: rgb = np.flip(rgb, axis=0)
    elif method == 'T02B_D4_R50':
        assert code in range(8)
        rgb = np.rot90(rgb, k=code % 4, axes=(0, 1))
        if code >= 4: rgb = np.flip(rgb, axis=1)
    else:
        raise ValueError(method)
    return np.ascontiguousarray(rgb)

class T02Dataset(Dataset):
    def __init__(self, data, method, train=False):
        assert method in METHODS_T02
        self.data = data.reset_index(drop=True)
        self.method, self.train = method, train
    def __len__(self):
        return len(self.data)
    def __getitem__(self, index):
        row = self.data.iloc[index]
        with Image.open(row.path) as im:
            image = im.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
        if self.train:
            if self.method == 'T02A_HV_R50':
                code = int(torch.rand(()).item() < 0.5) + 2*int(torch.rand(()).item() < 0.5)
            else:
                code = int(torch.randint(8, ()).item())
            image = Image.fromarray(t02_geometry(np.asarray(image), self.method, code))
        return normalize(image), int(row.label)

def t02_loader(data, method, train=False):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(T02Dataset(data, method, train), batch_size=BATCH_SIZE,
        shuffle=train, num_workers=NUM_WORKERS, pin_memory=True,
        worker_init_fn=seed_worker, generator=generator, drop_last=False)

def t02_preflight():
    # Kiểm chức năng B00 vẫn nguyên, trước khi chuyển loader tạm thời.
    src = '\n'.join(inspect.getsource(f) for f in [reset_seed, SkinDataset.__init__,
        SkinDataset.__getitem__, SkinDataset.__len__, seed_worker, make_loader,
        build_model, metrics, evaluate, export_evaluation, train_experiment])
    assert hashlib.sha256(src.encode()).hexdigest() == reference['training_source_sha']
    assert sha_file(B00_ROOT/'B00_R50/best_resnet50.pt') == reference['checkpoint_sha']
    assert DEVICE.type == 'cuda', 'Bật GPU T4.'
    assert torch.cuda.get_device_name(0) == b00_config['environment']['gpu']
    for key, value in dict(labels=LABELS, seed=SEED, size=IMG_SIZE, batch=BATCH_SIZE,
        epochs=EPOCHS, patience=PATIENCE, workers=NUM_WORKERS, lr=LR,
        weight_decay=WEIGHT_DECAY).items():
        assert value == b00_config[key], key
    # Kiểm split hiện trong RAM, không chỉ dựa vào số lượng ảnh.
    locked = pd.read_csv(B00_ROOT/'split_manifest.csv', keep_default_na=False)
    cols = ['image','label','label_name','group','file_sha256','split']
    for split in ['train','val','test']:
        pd.testing.assert_frame_equal(
            parts[split][cols].sort_values('image').reset_index(drop=True).astype(str),
            locked.loc[locked.split.eq(split), cols].sort_values('image').reset_index(drop=True).astype(str))
    # Xác nhận tensor validation mới giống pipeline B00, không augmentation.
    sample = parts['val'].groupby('label', group_keys=False).head(2)
    original = SkinDataset(sample, 'B00_R50')
    for method in METHODS_T02:
        candidate = T02Dataset(sample, method, train=False)
        for i in range(len(sample)):
            x, y = original[i]; z, t = candidate[i]
            assert y == t and torch.equal(x, z), 'Validation khác B00.'
    reset_seed(); probe = build_model(); h = hashlib.sha256()
    for key, value in probe.state_dict().items():
        h.update(key.encode()); h.update(value.detach().cpu().numpy().tobytes())
    assert h.hexdigest() == b00_result['initial_sha'], 'Khởi tạo khác B00.'
    del probe; gc.collect(); torch.cuda.empty_cache()

def run_t02(method):
    assert method in METHODS_T02
    t02_preflight()
    branch = T02_ROOT/method
    branch.mkdir(exist_ok=True)
    source = '\n\n'.join(inspect.getsource(f) for f in
        [t02_geometry, T02Dataset.__init__, T02Dataset.__len__,
         T02Dataset.__getitem__, t02_loader, t02_preflight, run_t02])
    spec = {'T02A_HV_R50': 'independent HFlip(p=0.5), VFlip(p=0.5)',
            'T02B_D4_R50': 'uniform code 0..7: rot90(k=code%4), then HFlip if code>=4'}
    config = dict(method=method, reference=reference, baseline_config=b00_config,
        candidate_augmentation=spec[method], augmentation_stage='train only, after RGB resize, before normalization',
        loss='CrossEntropyLoss(weight=None,label_smoothing=0)',
        validation='resize+normalize B00, single view, NO TTA',
        train_from='ImageNet V1; same initial state as B00; NOT B00 checkpoint',
        source_sha=hashlib.sha256(source.encode()).hexdigest(), environment=env)
    next_lock(branch/'candidate_protocol_lock.json', config)
    (branch/'candidate_source.py').write_text(source, encoding='utf-8')
    # Các hàm train/eval gốc tra make_loader và criterion qua globals.
    keys = ['OUT','LOCK_SHA','resume','make_loader','criterion']
    old = {key: globals()[key] for key in keys}
    try:
        globals().update(OUT=branch, LOCK_SHA=sha_file(branch/'candidate_protocol_lock.json'),
            resume=None, make_loader=t02_loader, criterion=nn.CrossEntropyLoss())
        result = train_experiment(method)
    finally:
        globals().update(old)
        gc.collect(); torch.cuda.empty_cache()
    assert result['initial_sha'] == b00_result['initial_sha']
    folder = branch/method
    scores = next_report(pd.read_csv(folder/'val_predictions.csv'), branch/'comparison')
    for key, value in scores.items():
        assert np.isclose(value, result[key], rtol=0, atol=1e-10)
    dump_json(branch/'budget_note.json', dict(
        best_epoch=result['best_epoch'], completed_epochs=result['completed_epochs'],
        best_in_last_two_budget_epochs=result['best_epoch'] >= EPOCHS-1,
        interpretation='Late best suggests checking budget limits; does not justify increasing epochs for this branch alone.'))
    print(method, result)
    return result

def t02_bootstrap(predictions, n_boot=2000, seed=20261007):
    names = list(predictions)
    base = predictions[names[0]].sort_values('image').reset_index(drop=True)
    required = ['image','group','file_sha256','true_label']
    assert base.image.is_unique and base.groupby('group').true_label.nunique().max() == 1
    aligned = []
    for name in names:
        df = predictions[name].sort_values('image').reset_index(drop=True)
        assert df.image.is_unique
        pd.testing.assert_frame_equal(base[required].astype(str), df[required].astype(str))
        ids = df.pred_label.map(label_to_id)
        assert ids.notna().all()
        aligned.append(ids.to_numpy(dtype=int))
    y = base.true_label.map(label_to_id).to_numpy(dtype=int)
    codes, groups = pd.factorize(base.group, sort=True)
    group_y = np.array([y[np.flatnonzero(codes == g)[0]] for g in range(len(groups))])
    strata = [np.flatnonzero(group_y == k) for k in range(8)]
    assert all(len(s) > 0 for s in strata)
    # Mỗi group giữ mọi ảnh của nó. Chỉ lấy lại group, không lấy lại từng ảnh độc lập.
    counts = np.zeros((len(names), len(groups), 8), dtype=np.int64)
    for model_id, pred in enumerate(aligned):
        np.add.at(counts[model_id], (codes, pred), 1)
    def macro(cm):
        tp = np.diagonal(cm, axis1=-2, axis2=-1)
        denom = cm.sum(axis=-1) + cm.sum(axis=-2)
        return np.divide(2*tp, denom, out=np.zeros_like(tp, dtype=float), where=denom>0).mean(axis=-1)
    observed_cm = np.stack([confusion_matrix(y, pred, labels=range(8)) for pred in aligned])
    observed = macro(observed_cm)
    rng = np.random.default_rng(seed)
    draws = np.empty((n_boot, len(names)))
    for b in range(n_boot):
        cm = np.zeros((len(names), 8, 8), dtype=np.int64)
        for label, ids in enumerate(strata):
            chosen = rng.choice(ids, size=len(ids), replace=True)
            cm[:, label, :] = counts[:, chosen, :].sum(axis=1)
        draws[b] = macro(cm)
    pairs = [(i,0) for i in range(1,len(names))]
    if len(names) == 3: pairs.append((2,1))  # B so với A, bổ sung mô tả.
    records = []
    for i,j in pairs:
        delta = 100*(draws[:,i]-draws[:,j])
        lo,hi = np.quantile(delta, [0.025,0.975])
        records.append(dict(candidate=names[i], reference=names[j],
            delta_macro_f1_pp=100*(observed[i]-observed[j]),
            ci95_low_pp=float(lo), ci95_high_pp=float(hi), n_boot=n_boot,
            contains_zero=bool(lo<=0<=hi)))
    group_support = pd.DataFrame(dict(label=LABELS, n_groups=[len(s) for s in strata]))
    return pd.DataFrame(records), pd.DataFrame(draws,columns=names), group_support

src = '\n\n'.join(inspect.getsource(f) for f in [t02_geometry, T02Dataset.__init__, T02Dataset.__len__,
      T02Dataset.__getitem__, t02_loader, t02_preflight, run_t02])
check('T02B_candidate_source_sha', hashlib.sha256(src.encode()).hexdigest() == cand_lock['source_sha'] == EXPECTED['t02b_source'])
check('t02_bootstrap_equals_analysis_source', inspect.getsource(t02_bootstrap) in (T02_IN / 'analysis_source.py').read_text())


PASS T02B_candidate_source_sha 
PASS t02_bootstrap_equals_analysis_source 


## L01 — đối chứng T02B, guard NaN/Inf (chép nguyên văn P04SOG v1.1), trọng số, kiểm loss, hàm train, pipeline/khởi tạo

In [8]:
# L01 — xác lập đối chứng T02B_D4_R50 (CE không trọng số). KHÔNG dùng SoG làm đối chứng.
for rel, sha in EXPECTED['t02b_extra'].items():
    hashes = sorted({sha_file(r / rel) if (r / rel).is_file() else 'MISSING' for r in roots})
    check(f'sha:{rel}', hashes == [sha], hashes)
bc = cand_lock['baseline_config']
check('control_lock_loss_unweighted_ce', cand_lock['loss'] == 'CrossEntropyLoss(weight=None,label_smoothing=0)'
      and bc['loss'] == cand_lock['loss'], (cand_lock['loss'], bc['loss']))
check('control_sampler_no_weighting', bc['sampler'] == 'shuffle, no weighting', bc['sampler'])
HP = dict(seed=42, size=224, batch=32, epochs=16, patience=4, workers=2, lr=0.0001, weight_decay=0.0001, model='resnet50',
          weights='IMAGENET1K_V1', scheduler='CosineAnnealingLR T_max=16', checkpoint='max val macro-F1; min_delta=1e-5', amp=False,
          split_sha=EXPECTED['split'] if 'split' in EXPECTED else bc['split_sha'])
check('control_lock_hparams', all(bc[k] == v for k, v in HP.items()) and bc['split_sha'] == '38a5a9aca907d56685bddfcd874e162d8bf2c141a7247025410eeeee6bcf522a'
      and (SEED, IMG_SIZE, BATCH_SIZE, EPOCHS, PATIENCE) == (42, 224, 32, 16, 4), {k: bc.get(k) for k in HP})
check('val_rows_2539', len(parts['val']) == EXPECTED['n_val'] == len(t02b_pred), (len(parts['val']), len(t02b_pred)))
check('train_rows_17723', len(parts['train']) == EXPECTED['n_train'], len(parts['train']))
check('val_groups_1392', parts['val'].group.nunique() == EXPECTED['val_groups'], parts['val'].group.nunique())
ctrl_scores = metrics(t02b_pred.true_label.map(label_to_id), t02b_pred.pred_label.map(label_to_id))
check('T02B_complete_metrics_equal_predictions', all(np.isclose(ctrl_scores[k], t02b_result[k], rtol=0, atol=1e-10) for k in ctrl_scores), ctrl_scores)
t02b_hist = pd.read_csv(T02_IN / 'T02B_D4_R50/T02B_D4_R50/history.csv')
check('T02B_history_16_epochs_best14', len(t02b_hist) == t02b_result['completed_epochs'] == 16 and t02b_result['best_epoch'] == 14)


PASS sha:T02B_D4_R50/T02B_D4_R50/history.csv ['a58f2b866d5bf81095d6d9d2904889840706b3869b4cbac32bc80811c0bf7bf5']
PASS sha:T02B_D4_R50/T02B_D4_R50/val_per_class.csv ['d72e6f1891378ce16ef25d96f0aa110d0fbd0f71c27a1e93897aa7e4984041e2']
PASS sha:T02B_D4_R50/T02B_D4_R50/val_confusion.csv ['98301d70d06aa4140bc6725c0f6013bb89b9a783d580e6856ef26a95d30fffbc']
PASS sha:T02B_D4_R50/T02B_D4_R50/val_metrics.json ['bba570b85662784a26fa8f06797b152ac9d5c768fdb21f8c4a3dc46f901b368a']
PASS sha:bootstrap_group_support.csv ['9683f41ccf7280280249302b98f10e38babc504efc56cde6c9dd8001918035bb']
PASS control_lock_loss_unweighted_ce ('CrossEntropyLoss(weight=None,label_smoothing=0)', 'CrossEntropyLoss(weight=None,label_smoothing=0)')
PASS control_sampler_no_weighting shuffle, no weighting
PASS control_lock_hparams {'seed': 42, 'size': 224, 'batch': 32, 'epochs': 16, 'patience': 4, 'workers': 2, 'lr': 0.0001, 'weight_decay': 0.0001, 'model': 'resnet50', 'weights': 'IMAGENET1K_V1', 'scheduler': 'CosineAnnealingL

In [9]:
# GUARD (P04SOG_CLIPPING_QA): train_experiment GỐC ở cell VERBATIM_B00 giữ nguyên để khớp hash training_source 8399...
# Bản có guard là hàm MỚI train_experiment_guarded, có hash MỚI (EXPECTED['training_source_guarded']); diff: diffs/train_experiment_guard.diff
import math

def assert_finite_loss(loss, where):
    # Chỉ đọc giá trị; không sửa tensor loss. Gọi TRƯỚC backward/optimizer.step.
    value = float(loss.detach().item()) if hasattr(loss, 'detach') else float(loss)
    if not math.isfinite(value):
        raise FloatingPointError(f'STOP_REPORT_TO_CHA: loss không hữu hạn ({value}) tại {where}; dừng trước backward/optimizer.step.')

def assert_finite_metrics(val_loss, scores, where):
    values = dict(val_loss=val_loss, **scores)
    bad = {k: v for k, v in values.items() if not math.isfinite(float(v))}
    if bad:
        raise FloatingPointError(f'STOP_REPORT_TO_CHA: validation không hữu hạn {bad} tại {where}.')

def train_experiment_guarded(method):
    assert DEVICE.type == 'cuda', 'Bật GPU; không train ResNet50 trên CPU.'
    folder = OUT / method
    if not folder.exists() and resume and (resume / method / 'complete.json').exists():
        shutil.copytree(resume / method, folder)
    folder.mkdir(exist_ok=True)
    complete_path = folder / 'complete.json'
    if complete_path.exists():
        saved = json.loads(complete_path.read_text())
        assert saved['lock_sha'] == LOCK_SHA
        assert sha_file(folder / 'best_resnet50.pt') == saved['checkpoint_sha']
        print(method, 'đã hoàn thành; tái sử dụng kết quả, không train lại.')
        return saved
    if (folder / 'history.csv').exists():
        raise RuntimeError('Lượt train trước chưa hoàn thành. Giữ output đó; dùng OUT mới để chạy lại cả protocol, không âm thầm resume optimizer.')
    reset_seed()
    model = build_model()
    # Hash khởi tạo: xác minh hai lượt khởi tạo giống nhau.
    initial = hashlib.sha256()
    for key, value in model.state_dict().items():
        initial.update(key.encode()); initial.update(value.detach().cpu().numpy().tobytes())
    init_sha = initial.hexdigest()
    train_loader = make_loader(parts['train'], method, train=True)
    val_loader = make_loader(parts['val'], method)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best, wait, history = -1.0, 0, []
    started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train(); loss_sum = 0.0; total = 0
        for images, y in train_loader:
            images, y = images.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, y)
            assert_finite_loss(loss, f'{method} epoch {epoch} train batch')   # GUARD: dừng TRƯỚC backward/optimizer.step
            loss.backward(); optimizer.step()
            loss_sum += loss.item() * len(y); total += len(y)
        val_loss, scores, _, _, _ = evaluate(model, val_loader)
        assert_finite_metrics(val_loss, scores, f'{method} epoch {epoch} validation')   # GUARD: validation loss + metrics
        history.append(dict(epoch=epoch, train_loss=loss_sum/total, val_loss=val_loss,
                            lr=optimizer.param_groups[0]['lr'], **scores))
        pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
        print(f'{method} epoch {epoch:02d}: train={loss_sum/total:.4f}, val={val_loss:.4f}, F1={scores["macro_f1"]:.4f}')
        if scores['macro_f1'] > best + 1e-5:
            best, best_epoch, wait = scores['macro_f1'], epoch, 0
            torch.save(model.state_dict(), folder / 'best_resnet50.pt')
        else:
            wait += 1
        scheduler.step()
        if wait >= PATIENCE: break
    model.load_state_dict(torch.load(folder / 'best_resnet50.pt', map_location=DEVICE, weights_only=True))
    scores = export_evaluation(model, parts['val'], method, folder, 'val')
    assert_finite_metrics(json.loads((folder / 'val_metrics.json').read_text())['loss'], scores, f'{method} final val export')   # GUARD
    h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(h.epoch, h.train_loss, label='train'); axes[0].plot(h.epoch, h.val_loss, label='val'); axes[0].legend()
    axes[1].plot(h.epoch, h.macro_f1); axes[1].set_title('Validation macro-F1')
    for ax in axes: ax.set_xlabel('Epoch'); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(folder / 'learning_curves.png', dpi=120); plt.close(fig)
    saved = dict(method=method, lock_sha=LOCK_SHA, initial_sha=init_sha,
        checkpoint_sha=sha_file(folder / 'best_resnet50.pt'), best_epoch=best_epoch,
        completed_epochs=len(history), seconds=time.perf_counter()-started, **scores)
    dump_json(complete_path, saved)
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()
    return saved

TRAINING_GUARDED_SOURCE = '\n'.join(inspect.getsource(f) for f in [reset_seed, SkinDataset.__init__, SkinDataset.__getitem__, SkinDataset.__len__,
      seed_worker, make_loader, build_model, metrics, evaluate, export_evaluation, assert_finite_loss, assert_finite_metrics, train_experiment_guarded])
TRAINING_GUARDED_SHA = hashlib.sha256(TRAINING_GUARDED_SOURCE.encode()).hexdigest()
check('training_source_guarded_sha_new', TRAINING_GUARDED_SHA == EXPECTED['training_source_guarded'] and TRAINING_GUARDED_SHA != EXPECTED['training_source'], TRAINING_GUARDED_SHA)

def guard_tests():
    # Kiểm thử giả với tensor torch trên CPU; khôi phục RNG torch sau khi tạo Linear.
    state = torch.get_rng_state().clone(); res = {}
    lin = nn.Linear(2, 1); w0 = lin.weight.detach().clone()
    opt = torch.optim.SGD(lin.parameters(), lr=0.1)
    for name, bad in [('nan', float('nan')), ('inf', float('inf')), ('-inf', float('-inf'))]:
        opt.zero_grad(set_to_none=True)
        loss = lin(torch.ones(1, 2)).sum() * 0 + bad
        try:
            assert_finite_loss(loss, 'guard_test'); loss.backward(); opt.step(); res[name] = 'NOT_STOPPED'
        except FloatingPointError:
            res[name] = 'STOPPED_BEFORE_BACKWARD' if lin.weight.grad is None and torch.equal(lin.weight, w0) else 'STOPPED_TOO_LATE'
    loss = lin(torch.ones(1, 2)).sum(); before = loss.detach().clone()
    assert_finite_loss(loss, 'guard_test')
    res['finite_loss_unchanged'] = bool(torch.equal(loss.detach(), before)) and loss.requires_grad
    for name, (vl, sc) in dict(ok=(0.5, dict(accuracy=0.7, macro_f1=0.6, balanced_accuracy=0.5)), nan_val_loss=(float('nan'), dict(accuracy=0.7)),
                               inf_metric=(0.5, dict(macro_f1=float('inf'))), nan_metric=(0.5, dict(balanced_accuracy=float('nan')))).items():
        try: assert_finite_metrics(vl, sc, 'guard_test'); res['metrics_' + name] = 'PASSED_THROUGH'
        except FloatingPointError: res['metrics_' + name] = 'STOPPED'
    torch.set_rng_state(state)
    return res
GUARD_RESULT = guard_tests()
check('guard_stops_nonfinite_before_backward_torch_synthetic', all(GUARD_RESULT[k] == 'STOPPED_BEFORE_BACKWARD' for k in ['nan', 'inf', '-inf']), GUARD_RESULT)
check('guard_finite_loss_unchanged_torch_synthetic', GUARD_RESULT['finite_loss_unchanged'] is True, GUARD_RESULT)
check('guard_validation_loss_and_metrics_synthetic', GUARD_RESULT['metrics_ok'] == 'PASSED_THROUGH' and all(GUARD_RESULT['metrics_' + k] == 'STOPPED' for k in ['nan_val_loss', 'inf_metric', 'nan_metric']), GUARD_RESULT)


PASS training_source_guarded_sha_new 2a10a660d2a4d1304914e344c110c1a2cf3440fbfb49c2c29df946f41fe53f82
PASS guard_stops_nonfinite_before_backward_torch_synthetic {'nan': 'STOPPED_BEFORE_BACKWARD', 'inf': 'STOPPED_BEFORE_BACKWARD', '-inf': 'STOPPED_BEFORE_BACKWARD', 'finite_loss_unchanged': True, 'metrics_ok': 'PASSED_THROUGH', 'metrics_nan_val_loss': 'STOPPED', 'metrics_inf_metric': 'STOPPED', 'metrics_nan_metric': 'STOPPED'}
PASS guard_finite_loss_unchanged_torch_synthetic {'nan': 'STOPPED_BEFORE_BACKWARD', 'inf': 'STOPPED_BEFORE_BACKWARD', '-inf': 'STOPPED_BEFORE_BACKWARD', 'finite_loss_unchanged': True, 'metrics_ok': 'PASSED_THROUGH', 'metrics_nan_val_loss': 'STOPPED', 'metrics_inf_metric': 'STOPPED', 'metrics_nan_metric': 'STOPPED'}
PASS guard_validation_loss_and_metrics_synthetic {'nan': 'STOPPED_BEFORE_BACKWARD', 'inf': 'STOPPED_BEFORE_BACKWARD', '-inf': 'STOPPED_BEFORE_BACKWARD', 'finite_loss_unchanged': True, 'metrics_ok': 'PASSED_THROUGH', 'metrics_nan_val_loss': 'STOPPED', 'me

In [10]:
# L01 — trọng số lớp CHỈ từ parts['train'] (train manifest). Không đọc số lượng val/test.
train_counts = parts['train'].label_name.value_counts().reindex(LABELS)
check('train_counts_all_8_labels_present', train_counts.notna().all() and len(train_counts) == 8, train_counts.to_dict())
check('train_counts_nonzero', bool((train_counts.fillna(0) > 0).all()), train_counts.to_dict())   # n_c=0 -> STOP
man_counts = manifest[manifest.split.eq('train')].label_name.value_counts().reindex(LABELS)
check('train_counts_from_manifest_train_only', train_counts.equals(man_counts) and int(train_counts.sum()) == EXPECTED['n_train'], man_counts.to_dict())
check('train_counts_equal_expected', {k: int(v) for k, v in train_counts.items()} == EXPECTED['train_counts'], train_counts.to_dict())
n_c = train_counts.to_numpy(dtype=np.float64)
w_raw = 1.0 / np.sqrt(n_c)
CLASS_WEIGHTS = w_raw / w_raw.mean()          # w_c ∝ 1/sqrt(n_c), trung bình 8 trọng số = 1
check('weights_shape_8_finite_positive', CLASS_WEIGHTS.shape == (8,) and bool(np.isfinite(CLASS_WEIGHTS).all()) and bool((CLASS_WEIGHTS > 0).all()), CLASS_WEIGHTS.tolist())
check('weights_mean_equals_1', abs(float(CLASS_WEIGHTS.mean()) - 1.0) < 1e-12, float(CLASS_WEIGHTS.mean()))
prod = CLASS_WEIGHTS * np.sqrt(n_c)
check('weights_proportional_inv_sqrt', bool(np.allclose(prod, prod[0], rtol=1e-12, atol=0)), prod.tolist())
check('weights_equal_expected', bool(np.allclose(CLASS_WEIGHTS, [EXPECTED['class_weights'][l] for l in LABELS], rtol=0, atol=1e-12)), CLASS_WEIGHTS.tolist())
W_TENSOR = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32)   # dtype dùng khi train (float32, giống logits)
check('weights_float32_finite_positive', bool(torch.isfinite(W_TENSOR).all()) and bool((W_TENSOR > 0).all()), W_TENSOR.tolist())
weights_table = pd.DataFrame(dict(label_id=range(8), label=LABELS, n_train=train_counts.to_numpy(dtype=int),
    inv_sqrt_n=w_raw, weight=CLASS_WEIGHTS, weight_float32=W_TENSOR.numpy().astype(np.float64)))
weights_table.to_csv(OUT_ROOT / 'train_class_weights.csv', index=False)
AUDIT['class_weights'] = dict(zip(LABELS, CLASS_WEIGHTS.tolist()))
AUDIT['train_counts'] = {k: int(v) for k, v in train_counts.items()}
display(weights_table)


PASS train_counts_all_8_labels_present {'MEL': 3077, 'NV': 9019, 'BCC': 2374, 'AK': 611, 'BKL': 1847, 'DF': 187, 'VASC': 151, 'SCC': 457}
PASS train_counts_nonzero {'MEL': 3077, 'NV': 9019, 'BCC': 2374, 'AK': 611, 'BKL': 1847, 'DF': 187, 'VASC': 151, 'SCC': 457}
PASS train_counts_from_manifest_train_only {'MEL': 3077, 'NV': 9019, 'BCC': 2374, 'AK': 611, 'BKL': 1847, 'DF': 187, 'VASC': 151, 'SCC': 457}
PASS train_counts_equal_expected {'MEL': 3077, 'NV': 9019, 'BCC': 2374, 'AK': 611, 'BKL': 1847, 'DF': 187, 'VASC': 151, 'SCC': 457}
PASS weights_shape_8_finite_positive [0.4591693890646419, 0.26819920266119224, 0.5227527408409827, 1.0304242497848128, 0.5926567404688351, 1.8625840198638355, 2.072756101347629, 1.1914575559680705]
PASS weights_mean_equals_1 1.0
PASS weights_proportional_inv_sqrt [25.470453396175483, 25.470453396175483, 25.470453396175483, 25.47045339617548, 25.470453396175486, 25.470453396175486, 25.470453396175483, 25.470453396175483]
PASS weights_equal_expected [0.45916938

,label_id,label,n_train,inv_sqrt_n,weight,weight_float32
0,0,MEL,3077,0.018028,0.459169,0.459169
1,1,NV,9019,0.010530,0.268199,0.268199
2,2,BCC,2374,0.020524,0.522753,0.522753
3,3,AK,611,0.040456,1.030424,1.030424
4,4,BKL,1847,0.023268,0.592657,0.592657
5,5,DF,187,0.073127,1.862584,1.862584
6,6,VASC,151,0.081379,2.072756,2.072756
7,7,SCC,457,0.046778,1.191458,1.191458


In [11]:
# L01 — kiểm loss: (a) weight = 1 khớp CE thường cả loss lẫn gradient; (b) CE có trọng số khớp công thức
# reduction='mean' của PyTorch: sum_i w[y_i]*nll_i / sum_i w[y_i]. Logits/nhãn gi���, generator riêng (không chạm RNG toàn cục).
def make_train_criterion(device):
    return nn.CrossEntropyLoss(weight=W_TENSOR.to(device), reduction='mean', label_smoothing=0.0)

def loss_equivalence_tests():
    res = {}; ok = True
    g = torch.Generator().manual_seed(20261012)
    devices = ['cpu'] + (['cuda'] if torch.cuda.is_available() else [])
    for dev in devices:
        for dtype, tol_ones, tol_w in [(torch.float32, 1e-6, 1e-5), (torch.float64, 1e-12, 1e-10)]:
            key = f"{dev}_{str(dtype).split('.')[-1]}"
            logits0 = (torch.randn(257, 8, generator=g, dtype=torch.float64) * 3).to(dtype)
            y0 = torch.randint(0, 8, (257,), generator=g); y0[:8] = torch.arange(8)
            logits0, y = logits0.to(dev), y0.to(dev)
            a = logits0.clone().requires_grad_(True); b = logits0.clone().requires_grad_(True)
            la = nn.CrossEntropyLoss()(a, y)
            lb = nn.CrossEntropyLoss(weight=torch.ones(8, dtype=dtype, device=dev), reduction='mean', label_smoothing=0.0)(b, y)
            la.backward(); lb.backward()
            d_loss = float((la - lb).abs()); d_grad = float((a.grad - b.grad).abs().max())
            w = W_TENSOR.to(device=dev, dtype=dtype)
            c = logits0.clone().requires_grad_(True); d = logits0.clone().requires_grad_(True)
            lc = nn.CrossEntropyLoss(weight=w, reduction='mean', label_smoothing=0.0)(c, y)
            nll = -torch.log_softmax(d, 1).gather(1, y[:, None]).squeeze(1)
            ld = (w[y] * nll).sum() / w[y].sum()
            lc.backward(); ld.backward()
            dw_loss = float((lc - ld).abs()); dw_grad = float((c.grad - d.grad).abs().max())
            finite = all(bool(torch.isfinite(t).all()) for t in [la, lb, lc, ld, a.grad, b.grad, c.grad, d.grad])
            differs = float((lc - la).abs()) > 1e-4   # trọng số thực sự có hiệu lực
            res[key] = dict(ones_loss_absdiff=d_loss, ones_grad_maxabsdiff=d_grad, tol_ones=tol_ones,
                            weighted_vs_formula_loss_absdiff=dw_loss, weighted_vs_formula_grad_maxabsdiff=dw_grad, tol_weighted=tol_w,
                            finite=finite, weighted_differs_from_plain=differs)
            ok &= d_loss <= tol_ones and d_grad <= tol_ones and dw_loss <= tol_w and dw_grad <= tol_w and finite and differs
    return bool(ok), res

rng_before = torch.get_rng_state().clone()
LOSS_OK, LOSS_RES = loss_equivalence_tests()
check('loss_weight_ones_equals_plain_ce_loss_and_grad__weighted_equals_formula', LOSS_OK, LOSS_RES)
check('loss_tests_do_not_touch_global_rng', torch.equal(rng_before, torch.get_rng_state()))
tc = make_train_criterion('cpu')
check('train_criterion_config', isinstance(tc, nn.CrossEntropyLoss) and tc.reduction == 'mean' and tc.label_smoothing == 0.0
      and torch.equal(tc.weight, W_TENSOR) and tc.ignore_index == -100, dict(reduction=tc.reduction, ls=tc.label_smoothing))
check('validation_criterion_unweighted_ce', isinstance(criterion, nn.CrossEntropyLoss) and criterion.weight is None
      and criterion.label_smoothing == 0 and criterion.reduction == 'mean')
del tc


/tmp/ipykernel_23/2965558065.py:20: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  d_loss = float((la - lb).abs()); d_grad = float((a.grad - b.grad).abs().max())


PASS loss_weight_ones_equals_plain_ce_loss_and_grad__weighted_equals_formula {'cpu_float32': {'ones_loss_absdiff': 0.0, 'ones_grad_maxabsdiff': 0.0, 'tol_ones': 1e-06, 'weighted_vs_formula_loss_absdiff': 4.76837158203125e-07, 'weighted_vs_formula_grad_maxabsdiff': 0.0, 'tol_weighted': 1e-05, 'finite': True, 'weighted_differs_from_plain': True}, 'cpu_float64': {'ones_loss_abs
PASS loss_tests_do_not_touch_global_rng 
PASS train_criterion_config {'reduction': 'mean', 'ls': 0.0}
PASS validation_criterion_unweighted_ce 


In [12]:
# L01 — loader và hàm train. Loader = t02_loader nhánh T02B NGUYÊN BẢN (ảnh, D4, RNG, sampler, batch).
# Hàm train = train_experiment_guarded (đã dùng ở P04SOG v1.1) chỉ đổi DUY NHẤT dòng loss train; được kiểm bằng so khớp chuỗi.
def l01_loader(data, method, train=False):
    assert method == METHOD
    return t02_loader(data, CONTROL, train)

def train_experiment_l01(method):
    assert DEVICE.type == 'cuda', 'Bật GPU; không train ResNet50 trên CPU.'
    folder = OUT / method
    if not folder.exists() and resume and (resume / method / 'complete.json').exists():
        shutil.copytree(resume / method, folder)
    folder.mkdir(exist_ok=True)
    complete_path = folder / 'complete.json'
    if complete_path.exists():
        saved = json.loads(complete_path.read_text())
        assert saved['lock_sha'] == LOCK_SHA
        assert sha_file(folder / 'best_resnet50.pt') == saved['checkpoint_sha']
        print(method, 'đã hoàn thành; tái sử dụng kết quả, không train lại.')
        return saved
    if (folder / 'history.csv').exists():
        raise RuntimeError('Lượt train trước chưa hoàn thành. Giữ output đó; dùng OUT mới để chạy lại cả protocol, không âm thầm resume optimizer.')
    reset_seed()
    model = build_model()
    # Hash khởi tạo: xác minh hai lượt khởi tạo giống nhau.
    initial = hashlib.sha256()
    for key, value in model.state_dict().items():
        initial.update(key.encode()); initial.update(value.detach().cpu().numpy().tobytes())
    init_sha = initial.hexdigest()
    train_loader = make_loader(parts['train'], method, train=True)
    val_loader = make_loader(parts['val'], method)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best, wait, history = -1.0, 0, []
    started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train(); loss_sum = 0.0; total = 0
        for images, y in train_loader:
            images, y = images.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = train_criterion(logits, y)   # L01: CHỈ loss train có trọng số; evaluate() vẫn dùng criterion = CE thường
            assert_finite_loss(loss, f'{method} epoch {epoch} train batch')   # GUARD: dừng TRƯỚC backward/optimizer.step
            loss.backward(); optimizer.step()
            loss_sum += loss.item() * len(y); total += len(y)
        val_loss, scores, _, _, _ = evaluate(model, val_loader)
        assert_finite_metrics(val_loss, scores, f'{method} epoch {epoch} validation')   # GUARD: validation loss + metrics
        history.append(dict(epoch=epoch, train_loss=loss_sum/total, val_loss=val_loss,
                            lr=optimizer.param_groups[0]['lr'], **scores))
        pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
        print(f'{method} epoch {epoch:02d}: train={loss_sum/total:.4f}, val={val_loss:.4f}, F1={scores["macro_f1"]:.4f}')
        if scores['macro_f1'] > best + 1e-5:
            best, best_epoch, wait = scores['macro_f1'], epoch, 0
            torch.save(model.state_dict(), folder / 'best_resnet50.pt')
        else:
            wait += 1
        scheduler.step()
        if wait >= PATIENCE: break
    model.load_state_dict(torch.load(folder / 'best_resnet50.pt', map_location=DEVICE, weights_only=True))
    scores = export_evaluation(model, parts['val'], method, folder, 'val')
    assert_finite_metrics(json.loads((folder / 'val_metrics.json').read_text())['loss'], scores, f'{method} final val export')   # GUARD
    h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(h.epoch, h.train_loss, label='train'); axes[0].plot(h.epoch, h.val_loss, label='val'); axes[0].legend()
    axes[1].plot(h.epoch, h.macro_f1); axes[1].set_title('Validation macro-F1')
    for ax in axes: ax.set_xlabel('Epoch'); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(folder / 'learning_curves.png', dpi=120); plt.close(fig)
    saved = dict(method=method, lock_sha=LOCK_SHA, initial_sha=init_sha,
        checkpoint_sha=sha_file(folder / 'best_resnet50.pt'), best_epoch=best_epoch,
        completed_epochs=len(history), seconds=time.perf_counter()-started, **scores)
    dump_json(complete_path, saved)
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()
    return saved

OLD_LINE = '            loss = criterion(logits, y)\n'
NEW_LINE = '            loss = train_criterion(logits, y)   # L01: CHỈ loss train có trọng số; evaluate() vẫn dùng criterion = CE thường\n'
g_src, l_src = inspect.getsource(train_experiment_guarded), inspect.getsource(train_experiment_l01)
expected_l01 = g_src.replace('def train_experiment_guarded(method):', 'def train_experiment_l01(method):', 1).replace(OLD_LINE, NEW_LINE, 1)
check('l01_train_equals_guarded_except_loss_line', g_src.count(OLD_LINE) == 1 and l_src == expected_l01)
diff_l01 = list(difflib.unified_diff(g_src.splitlines(True), l_src.splitlines(True), 'train_experiment_guarded', 'train_experiment_l01'))
changed = [x for x in diff_l01 if x[:1] in '+-' and not x.startswith(('+++', '---'))]
check('l01_train_diff_only_name_and_loss_line', len(changed) == 4, changed)
o_src = inspect.getsource(train_experiment)
diff_guard = list(difflib.unified_diff(o_src.splitlines(True), g_src.splitlines(True), 'train_experiment (T02B, 8399…)', 'train_experiment_guarded'))
(OUT_ROOT / 'diff_train_T02B_to_guarded.diff').write_text(''.join(diff_guard), encoding='utf-8')
(OUT_ROOT / 'diff_train_guarded_to_L01.diff').write_text(''.join(diff_l01), encoding='utf-8')
added_guard = [x for x in diff_guard if x.startswith('+') and not x.startswith('+++')]
check('guard_diff_only_adds_finite_checks', all('assert_finite' in x or x.strip() in ('+', '+def train_experiment_guarded(method):') for x in added_guard)
      and all(x.startswith('-def train_experiment(') for x in diff_guard if x.startswith('-') and not x.startswith('---')), added_guard)
ev_src = inspect.getsource(evaluate)
check('evaluate_uses_unweighted_criterion', 'criterion(logits, y).item()' in ev_src and 'train_criterion' not in ev_src)
check('no_weighted_sampler_or_sog_in_l01', all(k not in l_src + inspect.getsource(l01_loader) for k in ['Weighted' + 'RandomSampler', 'sog', 'SOG']))
L01_SOURCE = '\n\n'.join(inspect.getsource(f) for f in [make_train_criterion, l01_loader, train_experiment_l01])
L01_SOURCE_SHA = hashlib.sha256(L01_SOURCE.encode()).hexdigest()
AUDIT['l01_source_sha'] = L01_SOURCE_SHA
AUDIT['training_source_guarded_sha'] = TRAINING_GUARDED_SHA


PASS l01_train_equals_guarded_except_loss_line 
PASS l01_train_diff_only_name_and_loss_line ['-def train_experiment_guarded(method):\n', '+def train_experiment_l01(method):\n', '-            loss = criterion(logits, y)\n', '+            loss = train_criterion(logits, y)   # L01: CHỈ loss train có trọng số; evaluate() vẫn dùng criterion = CE thường\n']
PASS guard_diff_only_adds_finite_checks ['+def train_experiment_guarded(method):\n', "+            assert_finite_loss(loss, f'{method} epoch {epoch} train batch')   # GUARD: dừng TRƯỚC backward/optimizer.step\n", "+        assert_finite_metrics(val_loss, scores, f'{method} epoch {epoch} validation')   # GUARD: validation loss + metrics\n"
PASS evaluate_uses_unweighted_criterion 
PASS no_weighted_sampler_or_sog_in_l01 


In [13]:
# L01 — pipeline ảnh, RNG augmentation, loader và khởi tạo phải khớp đối chứng T02B.
def rng_snapshot():
    s = dict(py=random.getstate(), np=np.random.get_state(), torch=torch.get_rng_state().clone())
    if torch.cuda.is_available():
        s['cuda'] = [t.clone() for t in torch.cuda.get_rng_state_all()]
    return s
def rng_equal(a, b):
    ok = a['py'] == b['py'] and all(np.array_equal(x, y) if isinstance(x, np.ndarray) else x == y for x, y in zip(a['np'], b['np']))
    ok = ok and torch.equal(a['torch'], b['torch'])
    if 'cuda' in a: ok = ok and all(torch.equal(x, y) for x, y in zip(a['cuda'], b['cuda']))
    return bool(ok)

samp_val = parts['val'].groupby('label_name', sort=False).head(2).reset_index(drop=True)
samp_train = parts['train'].groupby('label_name', sort=False).head(2).reset_index(drop=True)
lv = l01_loader(samp_val, METHOD, False).dataset
ok_val = isinstance(lv, T02Dataset) and lv.method == CONTROL and lv.train is False and all(
    torch.equal(lv[i][0], SkinDataset(samp_val, 'B00_R50')[i][0]) and lv[i][1] == SkinDataset(samp_val, 'B00_R50')[i][1] for i in range(len(samp_val)))
check('val_pipeline_equals_T02B_and_B00_no_tta', bool(ok_val), len(samp_val))
ok_rng = True
for i in range(len(samp_train)):
    torch.manual_seed(1000 + i); a = T02Dataset(samp_train, CONTROL, True)[i][0]; sa = torch.get_rng_state().clone()
    torch.manual_seed(1000 + i); b = l01_loader(samp_train, METHOD, True).dataset[i][0]; sb = torch.get_rng_state().clone()
    ok_rng &= torch.equal(a, b) and torch.equal(sa, sb)
check('train_D4_augmentation_and_rng_draw_equal_T02B', bool(ok_rng), len(samp_train))
for tr in [True, False]:
    A, B = t02_loader(samp_train, CONTROL, tr), l01_loader(samp_train, METHOD, tr)
    same = (A.batch_size, A.num_workers, A.pin_memory, A.drop_last, type(A.sampler).__name__, A.generator.initial_seed(), A.worker_init_fn) == \
           (B.batch_size, B.num_workers, B.pin_memory, B.drop_last, type(B.sampler).__name__, B.generator.initial_seed(), B.worker_init_fn)
    check(f'loader_settings_equal_T02B_train_{tr}', same and A.batch_size == 32)
s0 = rng_snapshot()
g = torch.Generator().manual_seed(7)
_ = make_train_criterion('cpu')(torch.randn(32, 8, generator=g), torch.randint(0, 8, (32,), generator=g))
check('train_criterion_consumes_no_global_rng', rng_equal(s0, rng_snapshot()))
# Khởi tạo tái lập từ ImageNet V1 (cần Internet để tải trọng số torchvision).
reset_seed()
probe = build_model()
initial = hashlib.sha256()
for key, value in probe.state_dict().items():
    initial.update(key.encode()); initial.update(value.detach().cpu().numpy().tobytes())
check('initial_sha_reproduced_equals_T02B', initial.hexdigest() == EXPECTED['initial_sha'] == t02b_result['initial_sha'], initial.hexdigest())
del probe; gc.collect()
reset_seed()


PASS val_pipeline_equals_T02B_and_B00_no_tta 16
PASS train_D4_augmentation_and_rng_draw_equal_T02B 16
PASS loader_settings_equal_T02B_train_True 
PASS loader_settings_equal_T02B_train_False 
PASS train_criterion_consumes_no_global_rng 
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 211MB/s]


PASS initial_sha_reproduced_equals_T02B 37a03a16d86fda7f927000366760327e605b16fc8b4cb6a565a4ce763fe9003c


In [14]:
AUDIT['status'] = 'PREFLIGHT_PASS_KAGGLE'
AUDIT['environment'] = env_now
dump_json(OUT_ROOT / 'preflight_audit.json', json.loads(json.dumps(AUDIT, default=str)))
PREFLIGHT_SHA = sha_file(OUT_ROOT / 'preflight_audit.json')
lines = ['# L01_WCE_SQRT_T02B_R50 — preflight (Kaggle, dữ liệu thật)', '', f"Status: {AUDIT['status']}", f'preflight_audit.json SHA256: {PREFLIGHT_SHA}', '']
lines += [f"- {k}: {v['status']}" for k, v in AUDIT['checks'].items()]
lines += ['', 'Chưa huấn luyện. Gửi preflight_audit.json, train_class_weights.csv và 2 file diff cho CHA.']
(OUT_ROOT / 'PREFLIGHT_REPORT.md').write_text('\n'.join(lines), encoding='utf-8')
print('\n'.join(lines))
# Pha chạy: audit riêng, không ghi đè preflight_audit.json.
AUDIT_FILE = 'run_audit.json'
AUDIT = dict(task_id='L01_WCE_SQRT_T02B_R50', version='1.0', phase='run', method=METHOD, control=CONTROL,
             preflight_audit_sha=PREFLIGHT_SHA, status='RUNNING' if RUN_TRAIN else 'NOT_RUN_RUN_TRAIN_FALSE', checks={})


# L01_WCE_SQRT_T02B_R50 — preflight (Kaggle, dữ liệu thật)

Status: PREFLIGHT_PASS_KAGGLE
preflight_audit.json SHA256: 5dbfce68e68bcc57ceeed6fb1017a1e38bc8076f0831e1a8fc189b6ed44a684c

- t02_output_found: PASS
- sha:split_manifest.csv: PASS
- sha:B00_protocol_lock.json: PASS
- sha:B00_complete.json: PASS
- sha:reference.json: PASS
- sha:B00_val_predictions.csv: PASS
- sha:analysis_source.py: PASS
- sha:T02B_D4_R50/candidate_protocol_lock.json: PASS
- sha:T02B_D4_R50/candidate_source.py: PASS
- sha:T02B_D4_R50/T02B_D4_R50/complete.json: PASS
- sha:T02B_D4_R50/T02B_D4_R50/val_predictions.csv: PASS
- sha:CE/class_distribution.csv: PASS
- sha:CE/data_audit.json: PASS
- reference_points_to_lock_split: PASS
- candidate_lock_embeds_B00_lock: PASS
- T02B_complete_lock_init: PASS
- environment_libraries_equal_lock: PASS
- gpu_equals_lock: PASS
- notebook_config_equals_lock: PASS
- manifest_rows_columns: PASS
- label_order: PASS
- no_cross_split_image: PASS
- no_cross_split_file_sha256: PASS
- n

## Huấn luyện — CHỈ khi CHA duyệt preflight và đặt RUN_TRAIN=True (GPU Tesla T4)

In [15]:
if RUN_TRAIN:
    check('gpu_equals_lock_at_train', torch.cuda.is_available() and torch.cuda.get_device_name(0) == env_lock['gpu'])
    branch = OUT_ROOT / METHOD
    branch.mkdir(exist_ok=True)
    config = dict(method=METHOD, reference=reference, baseline_config=cand_lock['baseline_config'], control=CONTROL,
        control_lock_sha=EXPECTED['t02b_lock'], control_source_sha=EXPECTED['t02b_source'],
        candidate_augmentation=cand_lock['candidate_augmentation'], augmentation_stage=cand_lock['augmentation_stage'],
        train_loss="CrossEntropyLoss(weight=w, reduction='mean', label_smoothing=0); w_c ∝ 1/sqrt(n_c), n_c từ train manifest, mean(w)=1",
        train_loss_reduction_note="PyTorch mean có weight: sum_i w[y_i]*nll_i / sum_i w[y_i] theo từng batch",
        train_counts={k: int(v) for k, v in train_counts.items()}, class_weights=dict(zip(LABELS, CLASS_WEIGHTS.tolist())),
        class_weights_float32=W_TENSOR.tolist(),
        validation_loss='CrossEntropyLoss(weight=None,label_smoothing=0) — cùng định nghĩa với T02B',
        train_loss_comparability='train_loss trong history là CE có trọng số; KHÔNG so độ lớn với train_loss T02B để chọn phương án',
        validation=cand_lock['validation'], sampler='shuffle, no weighting', optimizer='AdamW(lr=1e-4, weight_decay=1e-4)',
        scheduler='CosineAnnealingLR T_max=16', checkpoint='max val macro-F1; min_delta=1e-5', epochs=EPOCHS, patience=PATIENCE,
        train_from='ImageNet V1; initial_sha phải bằng T02B; KHÔNG từ checkpoint T02B/SoG',
        l01_source_sha=L01_SOURCE_SHA, training_source_original_sha=EXPECTED['training_source'],
        training_source_guarded_sha=TRAINING_GUARDED_SHA, preflight_audit_sha=PREFLIGHT_SHA, environment=env_now)
    lock_path = branch / 'candidate_protocol_lock.json'
    check('fresh_branch_no_previous_lock_or_history', not lock_path.exists() and not (branch / METHOD / 'history.csv').exists())
    dump_json(lock_path, config)
    (branch / 'candidate_source.py').write_text(L01_SOURCE, encoding='utf-8')
    (branch / 'training_source_guarded.py').write_text(TRAINING_GUARDED_SOURCE, encoding='utf-8')
    keys = ['OUT', 'LOCK_SHA', 'resume', 'make_loader', 'criterion', 'train_criterion']
    old = {key: globals().get(key) for key in keys}
    try:
        globals().update(OUT=branch, LOCK_SHA=sha_file(lock_path), resume=None, make_loader=l01_loader,
                         criterion=nn.CrossEntropyLoss(), train_criterion=make_train_criterion(DEVICE))
        result = train_experiment_l01(METHOD)
    finally:
        globals().update(old)
        gc.collect(); torch.cuda.empty_cache()
    check('trained_initial_sha_equals_T02B', result['initial_sha'] == EXPECTED['initial_sha'], result['initial_sha'])
    print(result)
else:
    print('RUN_TRAIN=False: không huấn luyện.')


PASS gpu_equals_lock_at_train 
PASS fresh_branch_no_previous_lock_or_history 
L01_WCE_SQRT_T02B_R50 epoch 01: train=1.1455, val=0.7763, F1=0.5123
L01_WCE_SQRT_T02B_R50 epoch 02: train=0.8889, val=0.9165, F1=0.4992
L01_WCE_SQRT_T02B_R50 epoch 03: train=0.7700, val=0.9242, F1=0.5251
L01_WCE_SQRT_T02B_R50 epoch 04: train=0.6777, val=0.7868, F1=0.5293
L01_WCE_SQRT_T02B_R50 epoch 05: train=0.5882, val=0.7590, F1=0.5863
L01_WCE_SQRT_T02B_R50 epoch 06: train=0.5112, val=0.8075, F1=0.5498
L01_WCE_SQRT_T02B_R50 epoch 07: train=0.4351, val=0.8033, F1=0.5878
L01_WCE_SQRT_T02B_R50 epoch 08: train=0.3607, val=0.9118, F1=0.5547
L01_WCE_SQRT_T02B_R50 epoch 09: train=0.3017, val=0.7859, F1=0.5692
L01_WCE_SQRT_T02B_R50 epoch 10: train=0.2358, val=0.8536, F1=0.6033
L01_WCE_SQRT_T02B_R50 epoch 11: train=0.1867, val=0.8832, F1=0.5878
L01_WCE_SQRT_T02B_R50 epoch 12: train=0.1439, val=0.9469, F1=0.5682
L01_WCE_SQRT_T02B_R50 epoch 13: train=0.1109, val=0.9379, F1=0.5715
L01_WCE_SQRT_T02B_R50 epoch 14: train=

## Đánh giá validation + so sánh ghép cặp với T02B (chỉ khi đã train)

In [16]:
# Chỉ chạy khi đã có kết quả train. So ghép cặp với T02B bằng t02_bootstrap đã khóa (2000 draw, seed 20261007).
folder = OUT_ROOT / METHOD / METHOD
cand_file = folder / 'val_predictions.csv'
if cand_file.exists():
    res = json.loads((folder / 'complete.json').read_text())
    check('complete_lock_checkpoint_init', res['lock_sha'] == sha_file(OUT_ROOT / METHOD / 'candidate_protocol_lock.json')
          and res['checkpoint_sha'] == sha_file(folder / 'best_resnet50.pt') and res['initial_sha'] == EXPECTED['initial_sha'])
    hist = pd.read_csv(folder / 'history.csv')
    check('history_finite_budget', bool(np.isfinite(hist.drop(columns='epoch').to_numpy(dtype=float)).all())
          and 1 <= len(hist) <= EPOCHS and res['completed_epochs'] == len(hist), len(hist))
    n = len(hist)
    check('lr_schedule_equals_T02B_same_epochs', bool(np.allclose(hist.lr.to_numpy(), t02b_hist.lr.to_numpy()[:n], rtol=0, atol=1e-15)))
    best_idx = hist.macro_f1.to_numpy(); bests = []; b = -1.0; wait = 0
    for e, f in zip(hist.epoch, best_idx):
        if f > b + 1e-5: b, be, wait = f, e, 0
        else: wait += 1
    check('best_epoch_and_stop_rule_reproduced', be == res['best_epoch'] and (n == EPOCHS or wait >= PATIENCE), dict(best=be, n=n, wait=wait))
    cand = pd.read_csv(cand_file, keep_default_na=False).sort_values('image').reset_index(drop=True)
    v = parts['val'].rename(columns={'label_name': 'true_label'})
    ok = cand.image.is_unique and len(cand) == EXPECTED['n_val'] and list(cand.image) == list(v.image) and \
         all((cand[c].values == v[c].values).all() for c in ['group', 'file_sha256', 'true_label'])
    check('val_predictions_2539_match_manifest', bool(ok), len(cand))
    check('no_test_image_in_predictions', not (set(cand.image) & set(parts['test'].image)))
    P = cand[['p_' + l for l in LABELS]].to_numpy(dtype=float)
    check('probabilities_finite_sum1', bool(np.isfinite(P).all()) and bool(np.allclose(P.sum(1), 1, atol=1e-4)))
    check('pred_equals_argmax', (cand.pred_label.values == np.array(LABELS)[P.argmax(1)]).all())
    vm = json.loads((folder / 'val_metrics.json').read_text())
    sc = metrics(cand.true_label.map(label_to_id), cand.pred_label.map(label_to_id))
    check('metrics_finite_and_recomputed', all(math.isfinite(float(x)) for x in vm.values())
          and all(np.isclose(sc[k], vm[k], rtol=0, atol=1e-10) and np.isclose(sc[k], res[k], rtol=0, atol=1e-10) for k in sc), dict(vm))
    comp = OUT_ROOT / METHOD / 'comparison'; comp.mkdir(exist_ok=True)
    deltas, draws, support = t02_bootstrap({CONTROL: t02b_pred, METHOD: cand})
    check('bootstrap_groups_1392', int(support.n_groups.sum()) == EXPECTED['val_groups'] and
          support.equals(pd.read_csv(T02_IN / 'bootstrap_group_support.csv')), support.n_groups.tolist())
    deltas.to_csv(comp / 'bootstrap_delta_vs_T02B.csv', index=False)
    draws.to_csv(comp / 'bootstrap_draws.csv', index=False); support.to_csv(comp / 'bootstrap_group_support.csv', index=False)
    pc_c = pd.read_csv(T02_IN / 'T02B_D4_R50/T02B_D4_R50/val_per_class.csv', index_col=0).reindex(LABELS)
    pc_l = pd.read_csv(folder / 'val_per_class.csv', index_col=0).reindex(LABELS)
    check('support_equal_control', (pc_c.support.values == pc_l.support.values).all())
    per_class = pd.DataFrame(dict(label=LABELS, support=pc_l.support.astype(int).values,
        precision_T02B=pc_c.precision.values, precision_L01=pc_l.precision.values,
        recall_T02B=pc_c.recall.values, recall_L01=pc_l.recall.values,
        f1_T02B=pc_c['f1-score'].values, f1_L01=pc_l['f1-score'].values))
    per_class['delta_f1_pp'] = 100 * (per_class.f1_L01 - per_class.f1_T02B)
    per_class['contribution_macro_pp'] = per_class.delta_f1_pp / 8
    per_class.to_csv(comp / 'per_class_vs_T02B.csv', index=False)
    j = t02b_pred[['image', 'true_label', 'pred_label']].merge(cand[['image', 'pred_label']], on='image', suffixes=('_T02B', '_L01'), validate='one_to_one')
    a, b2 = j.pred_label_T02B.eq(j.true_label), j.pred_label_L01.eq(j.true_label)
    j['transition'] = np.select([a & b2, a & ~b2, ~a & b2], ['right_to_right', 'right_to_wrong', 'wrong_to_right'], default='wrong_to_wrong')
    j.to_csv(comp / 'paired_validation.csv', index=False)
    pd.crosstab(j.true_label, j.transition).reindex(LABELS, fill_value=0).to_csv(comp / 'transitions_by_class.csv')
    table = pd.DataFrame([dict(method=CONTROL, **ctrl_scores, best_epoch=t02b_result['best_epoch'], completed_epochs=t02b_result['completed_epochs']),
                          dict(method=METHOD, **sc, best_epoch=res['best_epoch'], completed_epochs=res['completed_epochs'])])
    table.to_csv(comp / 'comparison_validation.csv', index=False)
    d = deltas.iloc[0]
    verdict = ('ung_vien_can_lap_seed' if d.delta_macro_f1_pp >= 1.0 and d.ci95_low_pp > 0 else
               'khong_ghi_nhan_cai_thien_o_lan_chay_nay' if d.delta_macro_f1_pp <= 0 else 'chua_ket_luan')
    verdict_text = {'ung_vien_can_lap_seed': 'ứng viên cần lặp seed', 'khong_ghi_nhan_cai_thien_o_lan_chay_nay': 'không ghi nhận cải thiện ở lần chạy này', 'chua_ket_luan': 'chưa kết luận'}[verdict]
    dump_json(comp / 'decision.json', dict(rule='delta>=+1.0pp va CI95 thap>0 => ung vien can lap seed; delta<=0 => khong ghi nhan cai thien o lan chay nay; con lai chua ket luan',
        verdict=verdict, verdict_text=verdict_text, delta=d.to_dict(),
        ci_limit='CI chỉ phản ánh lấy mẫu lại group validation với 1 seed huấn luyện; không gồm biến thiên giữa seed hay chọn checkpoint.',
        loss_note='val_loss = CE không trọng số (cùng định nghĩa T02B); train_loss = CE có trọng số, không so độ lớn với T02B.'))
    AUDIT['status'] = 'RUN_COMPLETE_KAGGLE'; AUDIT['verdict'] = verdict
    dump_json(OUT_ROOT / 'run_audit.json', json.loads(json.dumps(AUDIT, default=str)))
    display(table); display(deltas); display(per_class); print(verdict_text)
    files = sorted(p for p in OUT_ROOT.rglob('*') if p.is_file() and p.name != 'SHA256SUMS.txt')
    (OUT_ROOT / 'SHA256SUMS.txt').write_text(''.join(f'{sha_file(p)}  {p.relative_to(OUT_ROOT)}\n' for p in files), encoding='utf-8')
    archive = OUT_ROOT.parent / (OUT_ROOT.name + '_reports.zip')
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT_ROOT.rglob('*')):
            if p.is_file() and p.suffix != '.pt':
                z.write(p, str(p.relative_to(OUT_ROOT)))
    print('Gửi ZIP reports + notebook đã chạy:', archive, '| .pt giữ trong FULL Output; hash có trong SHA256SUMS.txt')
else:
    print('Chưa có val_predictions của', METHOD, '- bỏ qua đánh giá.')


PASS complete_lock_checkpoint_init 
PASS history_finite_budget 14
PASS lr_schedule_equals_T02B_same_epochs 
PASS best_epoch_and_stop_rule_reproduced {'best': 10, 'n': 14, 'wait': 4}
PASS val_predictions_2539_match_manifest 2539
PASS no_test_image_in_predictions 
PASS probabilities_finite_sum1 
PASS pred_equals_argmax 
PASS metrics_finite_and_recomputed {'loss': 0.8535534162697899, 'accuracy': 0.7495076801890508, 'macro_f1': 0.6032509123244468, 'balanced_accuracy': 0.5995181692309809}
PASS bootstrap_groups_1392 [180, 837, 138, 32, 157, 10, 11, 27]
PASS support_equal_control 


,method,accuracy,macro_f1,balanced_accuracy,best_epoch,completed_epochs
0,T02B_D4_R50,0.769988,0.603002,0.586105,14,16
1,L01_WCE_SQRT_T02B_R50,0.749508,0.603251,0.599518,10,14


,candidate,reference,delta_macro_f1_pp,ci95_low_pp,ci95_high_pp,n_boot,contains_zero
0,L01_WCE_SQRT_T02B_R50,T02B_D4_R50,0.024907,-4.214227,4.594794,2000,True


,label,support,precision_T02B,precision_L01,recall_T02B,recall_L01,f1_T02B,f1_L01,delta_f1_pp,contribution_macro_pp
0,MEL,478,0.741546,0.696262,0.642259,0.623431,0.688341,0.657837,-3.050416,-0.381302
1,NV,1247,0.856383,0.885314,0.903769,0.860465,0.879438,0.872712,-0.672567,-0.084071
2,BCC,348,0.725191,0.696172,0.818966,0.836207,0.769231,0.759791,-0.943965,-0.117996
3,AK,84,0.394366,0.303279,0.333333,0.440476,0.361290,0.359223,-0.206702,-0.025838
4,BKL,285,0.644788,0.573944,0.585965,0.571930,0.613971,0.572935,-4.103561,-0.512945
5,DF,20,0.777778,0.533333,0.350000,0.400000,0.482759,0.457143,-2.561576,-0.320197
6,VASC,22,0.666667,0.850000,0.727273,0.772727,0.695652,0.809524,11.387164,1.423395
7,SCC,55,0.339623,0.400000,0.327273,0.290909,0.333333,0.336842,0.350877,0.043860


chưa kết luận
Gửi ZIP reports + notebook đã chạy: /kaggle/working/ISIC2019_R50_L01_WCE_SQRT_v1_0_reports.zip | .pt giữ trong FULL Output; hash có trong SHA256SUMS.txt
